In [1]:
import pandas as pd
import pymysql
from sqlalchemy import create_engine, URL
from getpass import getpass

In [2]:
password = getpass("Enter your MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="localhost",
    port=3306,
    database="sakila"
)

engine = create_engine(connection_url)

print("Connection engine created successfully.")

Enter your MySQL password:  ········


Connection engine created successfully.


In [4]:
test_query = """
SELECT *
FROM rental
LIMIT 5;
"""

test_df = pd.read_sql_query(test_query, engine)

test_df

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [6]:
from sqlalchemy import text

def rentals_month(engine, month, year):
    query = text("""
        SELECT *
        FROM rental
        WHERE MONTH(rental_date) = :month
          AND YEAR(rental_date) = :year;
    """)

    rentals_df = pd.read_sql_query(
        query,
        engine,
        params={"month": month, "year": year}
    )

    return rentals_df

In [7]:
rentals_may = rentals_month(engine, 5, 2005)

rentals_may.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [8]:
def rental_count_month(rentals_df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    rental_counts = (
        rentals_df
        .groupby("customer_id")
        .size()
        .reset_index(name=column_name)
    )

    return rental_counts

In [9]:
may_rental_counts = rental_count_month(
    rentals_may,
    5,
    2005
)

may_rental_counts.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [10]:
rentals_june = rentals_month(engine, 6, 2005)

rentals_june.head()


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


In [11]:
june_rental_counts = rental_count_month(
    rentals_june,
    6,
    2005
)

june_rental_counts.head()

,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [13]:
def compare_rentals(first_month_df, second_month_df):
    combined_df = pd.merge(
        first_month_df,
        second_month_df,
        on="customer_id",
        how="inner"
    )

    first_rentals_column = first_month_df.columns[1]
    second_rentals_column = second_month_df.columns[1]

    combined_df["difference"] = (
        combined_df[second_rentals_column]
        - combined_df[first_rentals_column]
    )

    return combined_df

In [14]:
may_june_comparison = compare_rentals(
    may_rental_counts,
    june_rental_counts
)

may_june_comparison.head()

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


In [15]:
print(
    "Customers active in both May and June:",
    len(may_june_comparison)
)

Customers active in both May and June: 512


In [16]:
activity_summary = pd.Series({
    "Increased activity": (may_june_comparison["difference"] > 0).sum(),
    "Decreased activity": (may_june_comparison["difference"] < 0).sum(),
    "Same activity": (may_june_comparison["difference"] == 0).sum()
})

activity_summary

Increased activity    358
Decreased activity     92
Same activity          62
dtype: int64